<a href="https://colab.research.google.com/github/Firefox-097/Steganography/blob/main/Experiment3_EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q kaggle

from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d marcozuppelli/stegoimagesdataset
!unzip -q stegoimagesdataset.zip -d /content/stegoimages

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/marcozuppelli/stegoimagesdataset
License(s): DbCL-1.0
100% 1.51G/1.51G [00:41<00:00, 39.3MB/s]



In [4]:
import os

print(os.listdir("/content/stegoimages"))

['dataset_information.csv', 'test', 'val', 'train']


In [5]:
import os

folders = [
    "/content/stegoimages/train/train/clean",
    "/content/stegoimages/train/train/stego",
    "/content/stegoimages/val/val/clean",
    "/content/stegoimages/val/val/stego",
    "/content/stegoimages/test/test/clean",
    "/content/stegoimages/test/test/stego"
]

for folder in folders:
    print(folder, ":", len(os.listdir(folder)))

/content/stegoimages/train/train/clean : 4000
/content/stegoimages/train/train/stego : 12000
/content/stegoimages/val/val/clean : 2000
/content/stegoimages/val/val/stego : 6000
/content/stegoimages/test/test/clean : 2000
/content/stegoimages/test/test/stego : 6000


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_gen = train_datagen.flow_from_directory(
    "/content/stegoimages/train/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    "/content/stegoimages/val/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 16000 images belonging to 2 classes.
Found 8000 images belonging to 2 classes.


In [7]:
print(train_gen.class_indices)
print("Training Images:", train_gen.samples)
print("Validation Images:", val_gen.samples)

{'clean': 0, 'stego': 1}
Training Images: 16000
Validation Images: 8000


In [8]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained backbone
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/best_stego_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_gen.classes

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(weights))

print(class_weights)

{0: np.float64(2.0), 1: np.float64(0.6666666666666666)}


In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ],
    class_weight=class_weights
)

Epoch 1/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5100 - loss: 0.6984
Epoch 1: val_accuracy improved from None to 0.25000, saving model to /content/drive/MyDrive/best_stego_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/best_stego_model.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 1960s 4s/step - accuracy: 0.4543 - loss: 0.6977 - val_accuracy: 0.2500 - val_loss: 0.6959 - learning_rate: 0.0010
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3115 - loss: 0.6943
Epoch 2: val_accuracy did not improve from 0.25000
500/500 ━━━━━━━━━━━━━━━━━━━━ 1947s 4s/step - accuracy: 0.4662 - loss: 0.6932 - val_accuracy: 0.2500 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3618 - loss: 0.6937
Epoch 3: val_accuracy did not improve from 0.25000
500/500 ━━━━━━━━━━━━━━━━━━━━ 1928s 4s/step - accuracy: 0.4248 - loss: 0.6932 - val_accuracy: 0.2500 - val_loss: 0.6946 - learning_rate: 0.0010
Epoch 4/20
500/500

In [ ]:
print("Train Accuracy:", history.history["accuracy"][-1])
print("Validation Accuracy:", history.history["val_accuracy"][-1])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    "/content/stegoimages/test/test",
    classes=["clean", "stego"],
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

In [ ]:
pred = model.predict(test_gen)

In [ ]:
import numpy as np
predictions = (pred > 0.5).astype(int).flatten()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(test_gen.classes, predictions)
disp = ConfusionMatrixDisplay(cm, display_labels=["Clean","Stego"])
disp.plot(cmap="Blues")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    test_gen.classes,
    predictions,
    target_names=["Clean","Stego"]
))